## carga inicial

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
# pip install pyarrow
# pip install brotli

# Changed encoding to 'latin-1' to handle special characters correctly
df = pd.read_csv("datasets/precios bencina/2016.csv", sep=';', decimal=',', encoding='utf-8', low_memory=False)
df

,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud
0,co1010701,JUAN RODOMIRO ANDRADE BARRIA 7275067-5 ...,COPEC,VICENTE PEREZ ROSALES,711,Llanquihue,Los Lagos,690,2016-01-01,Gasolina 93,"-41,26372706997888","-73,00196707248688"
1,sh1320201,COMERCIAL SANTA FE S.A.,SHELL,Concha y Toro,2464,Pirque,Metropolitana,752,2016-09-22,Gasolina 95,"-33,634001074731856","-70,57341456413269"
2,pb1410102,CARLOS RICARDO MORALES BASSAY,SURENERGY,Quinchilca,494,Los Lagos,Los Rí­os,714,2016-09-22,Gasolina 93,"-39,85955","-72,80795"
3,pe810110,Horacio Alejandro Espinoza Vergara Inversiones...,PETROBRAS,Av. Jorge Alessandri,277,Concepción,Bío Bío,691,2016-09-22,Gasolina 93,"-36,809593936900164","-73,08036804199219"
4,te1312001,SOC.COM. JERIA y CIA. LTDA.,SHELL,Pedro de Valdivia,3565,Ñuñoa,Metropolitana,709,2016-01-28,Gasolina 95,"-33,4554093275989","-70,60533553361893"
...,...,...,...,...,...,...,...,...,...,...,...,...
385168,co550201,ARROYO FERRAND CLAUDIA,COPEC,CARRERA,1513,Calera,Valparaíso,710,2016-10-20,Gasolina 93,"-32,7916286283671","-71,20279490947723"
385169,co510301,COMERCIAL PEREDO Y CIA LTDA,COPEC,AV. BORGOÑO,25175,Concón,Valparaíso,399,2016-01-28,Petroleo Diesel,"-32,92044810397277","-71,50862574577332"
385170,te911501,Sociedad Comercial LONCOADONAI Ltda,SHELL,Camino Internacional,2215,Pucón,Araucanía,720,2016-09-12,Gasolina 95,"-39,283382","-71,951132"
385171,sh1311904,GASTON POZO GONZALEZ,SHELL,Avenida Américo Vespucio,2025,Maipú,Metropolitana,466,2016-11-30,Petroleo Diesel,"-33,4712403703207","-70,75976371765137"


Lamentablemente, El archivo presenta un error de conversión irreversible en ciertos registros, como "ParcelaciÃ³n Patria Nueva sitio" y "Longitudinal sur nï¿½ 7-33 esquina ecuador.". Aunque el documento actual está estructurado y guardado correctamente en formato UTF-8, contiene cadenas de texto que provenían de una fuente con una codificación distinta (probablemente Latin-1 o Windows-1252) y que no fueron transformadas adecuadamente al integrarse.

Debido a esta mala conversión previa, el sistema no pudo interpretar los caracteres especiales originales (como tildes o la letra "ñ") y los sustituyó permanentemente por el carácter de reemplazo de error de Unicode (ï¿½), fenomeno llamado Mojibake. Dado que el código del carácter original fue borrado y sustituido por este indicador de error, el daño estructural no puede revertirse de forma automatizada y requiere una limpieza manual (ej. buscar y reemplazar) o una nueva extracción desde la base de datos de origen.

## transormacion a parquet

In [1]:
import pandas as pd
import os

# Directorio que contiene los archivos CSV
data_dir = "datasets/precios bencina"

# Lista para almacenar los DataFrames de cada archivo
dfs = []

# Obtener la lista de archivos CSV en el directorio
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

# Leer cada archivo CSV y agregarlo a la lista de DataFrames
for file in csv_files:
    file_path = os.path.join(data_dir, file)
    try:
        # Se leen en utf-8 dado que el archivo subyacente esta guardado en ese formato a pesar del mojibake
        df_temp = pd.read_csv(file_path, sep=';', decimal=',', encoding='utf-8', low_memory=False)
        dfs.append(df_temp)
    except Exception as e:
        print(f"Error al leer el archivo {file}: {e}")

# Unir todos los DataFrames en uno solo
if dfs:
    combined_df = pd.concat(dfs, ignore_index=True)

    # 1. ELIMINAR TODOS LOS TIPOS COMPLEJOS PRIMERO.
    # Convertir TODAS las columnas que no sean numéricas estrictas a 'string' puro de pandas.
    # Esto desintegra cualquier objeto 'Period' o 'Interval' que se haya formado en la concatenación
    # para evitar un "ArrowKeyError"
    for col in combined_df.columns:
        if combined_df[col].dtype == 'object' or str(combined_df[col].dtype).startswith('period') or str(combined_df[col].dtype).startswith('interval'):
            combined_df[col] = combined_df[col].astype("string")

    # 2. Ahora que el DataFrame está "limpio", convertimos la fecha a un datetime estándar.
    #if 'fecha_actualizacion' in combined_df.columns:
    #    combined_df['fecha_actualizacion'] = pd.to_datetime(combined_df['fecha_actualizacion'], errors='coerce')

    # --- Guardar en Parquet ---
    output_path = "datasets/precios bencina/precios_bencina.parquet"

    # Guardamos exclusivamente con compresión brotli
    combined_df.to_parquet(output_path, engine='pyarrow', compression='brotli')
    print(f"Archivos CSV combinados y guardados exitosamente con compresión brotli en {output_path}")

else:
    print("No se encontraron archivos CSV para procesar.")

Archivos CSV combinados y guardados exitosamente con compresión brotli en datasets/precios bencina/precios_bencina.parquet


In [ ]:
Los errores ArrowKeyError se producian por una incompatibilidad entre los tipos de datos especializados que Pandas infiere automáticamente y los que el motor PyArrow espera al escribir un archivo Parquet.
Al concatenar múltiples archivos CSV, Pandas creó columnas con tipos de datos complejos como Period (para fechas), Interval (para rangos) y object (para columnas con datos mixtos).
Conflicto: PyArrow, el motor para guardar en formato Parquet, es muy estricto con los tipos de datos que acepta. No pudo procesar estos tipos de datos especializados de Pandas, lo que generaba conflictos en su registro interno de tipos y lanzaba un error en multiples ocasiones.